# Netflix Exploratory Data Analysis (EDA)
**Role**: Senior Data Analyst / BI Developer  
**Project**: End-to-End Netflix Data Analysis  

This notebook explores the cleaned Netflix dataset (`cleaned_netflix.csv`) to uncover critical business insights and generate publication-quality visualizations for the portfolio. 
Insights include content growth patterns, geographical distribution of content, top genres, directors, rating analysis, and oldest/newest titles.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic styling (Dark/Premium theme matching Netflix color palette)
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['axes.facecolor'] = '#181818'
plt.rcParams['figure.facecolor'] = '#121212'
plt.rcParams['text.color'] = '#E5E5E5'
plt.rcParams['axes.labelcolor'] = '#E5E5E5'
plt.rcParams['xtick.color'] = '#E5E5E5'
plt.rcParams['ytick.color'] = '#E5E5E5'

netflix_colors = ['#E50914', '#221F1F', '#F5F5F1']
print("EDA environment initialized successfully!")

## 1. Load the Cleaned Data

In [ ]:
df = pd.read_csv('../data/cleaned_netflix.csv')
df.head()

## 2. Key Metrics Summary

In [ ]:
total_titles = df.shape[0]
movie_count = df[df['type'] == 'Movie'].shape[0]
tv_show_count = df[df['type'] == 'TV Show'].shape[0]
unique_countries = df['country'].nunique()

print(f"=== NETFLIX KEY METRICS ===")
print(f"Total Content Titles: {total_titles}")
print(f"Total Movies: {movie_count} ({movie_count/total_titles*100:.1f}%)")
print(f"Total TV Shows: {tv_show_count} ({tv_show_count/total_titles*100:.1f}%)")
print(f"Unique Production Countries: {unique_countries}")

## 3. Visual Analysis

### 3.1 Content Types Breakdown (Movie vs. TV Show)

In [ ]:
type_counts = df['type'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', colors=['#E50914', '#564d4d'], 
        textprops={'fontsize': 14, 'color': 'white'}, startangle=90, explode=[0, 0.05])
plt.title('Netflix Content Types: Movies vs TV Shows', fontsize=16, color='white', weight='bold')

os.makedirs('../dashboard_images', exist_ok=True)
plt.savefig('../dashboard_images/content_type_ratio.png', bbox_inches='tight', facecolor='#121212')
plt.show()

### 3.2 Content Growth over Years (Cumulative)

In [ ]:
growth_df = df.groupby(['year_added', 'type']).size().unstack(fill_value=0).cumsum()
plt.figure(figsize=(12, 6))
plt.plot(growth_df.index, growth_df['Movie'], color='#E50914', label='Movies', linewidth=3.5)
plt.plot(growth_df.index, growth_df['TV Show'], color='#B3B3B3', label='TV Shows', linewidth=3.5, linestyle='--')
plt.title('Netflix Content Growth Trajectory (Cumulative)', fontsize=16, color='white', weight='bold')
plt.xlabel('Year Added', fontsize=12, color='white')
plt.ylabel('Cumulative Number of Titles', fontsize=12, color='white')
plt.legend(facecolor='#181818', edgecolor='none', labelcolor='white')
plt.grid(color='#333333', linestyle=':')
plt.savefig('../dashboard_images/content_growth.png', bbox_inches='tight', facecolor='#121212')
plt.show()

### 3.3 Top 10 Countries Producing Content

In [ ]:
# Countries can be multi-valued (e.g. "United States, India"). We explode them to get a clean count
countries_series = df['country'].str.split(', ').dropna().explode()
countries_series = countries_series[countries_series != 'Unknown']
top_countries = countries_series.value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='Reds_r')
plt.title('Top 10 Countries Producing Netflix Content', fontsize=16, color='white', weight='bold')
plt.xlabel('Number of Titles', fontsize=12, color='white')
plt.ylabel('Country', fontsize=12, color='white')
plt.savefig('../dashboard_images/top_countries.png', bbox_inches='tight', facecolor='#121212')
plt.show()

### 3.4 Top 10 Directors

In [ ]:
directors_series = df['director'].str.split(', ').dropna().explode()
directors_series = directors_series[directors_series != 'Unknown']
top_directors = directors_series.value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_directors.values, y=top_directors.index, palette='Reds_r')
plt.title('Top 10 Most Frequent Directors on Netflix', fontsize=16, color='white', weight='bold')
plt.xlabel('Number of Titles Directed', fontsize=12, color='white')
plt.ylabel('Director', fontsize=12, color='white')
plt.savefig('../dashboard_images/top_directors.png', bbox_inches='tight', facecolor='#121212')
plt.show()

### 3.5 Top 10 Genres (listed_in)

In [ ]:
genres_series = df['listed_in'].str.split(', ').dropna().explode()
top_genres = genres_series.value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, palette='Reds_r')
plt.title('Top 10 Netflix Content Genres', fontsize=16, color='white', weight='bold')
plt.xlabel('Number of Titles', fontsize=12, color='white')
plt.ylabel('Genre', fontsize=12, color='white')
plt.savefig('../dashboard_images/top_genres.png', bbox_inches='tight', facecolor='#121212')
plt.show()

### 3.6 Ratings Distribution

In [ ]:
ratings_order = df['rating'].value_counts().index
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='rating', order=ratings_order, hue='type', palette=['#E50914', '#564d4d'])
plt.title('Distribution of Content Ratings on Netflix', fontsize=16, color='white', weight='bold')
plt.xlabel('Rating', fontsize=12, color='white')
plt.ylabel('Count', fontsize=12, color='white')
plt.legend(facecolor='#181818', edgecolor='none', labelcolor='white')
plt.xticks(rotation=45)
plt.savefig('../dashboard_images/ratings_distribution.png', bbox_inches='tight', facecolor='#121212')
plt.show()

### 3.7 Movie Duration Analysis

In [ ]:
movies_df = df[df['type'] == 'Movie'].dropna(subset=['duration_minutes'])
plt.figure(figsize=(12, 6))
sns.histplot(movies_df['duration_minutes'], kde=True, color='#E50914', bins=30)
plt.title('Distribution of Movie Durations', fontsize=16, color='white', weight='bold')
plt.xlabel('Duration (Minutes)', fontsize=12, color='white')
plt.ylabel('Frequency', fontsize=12, color='white')
plt.savefig('../dashboard_images/movie_duration_dist.png', bbox_inches='tight', facecolor='#121212')
plt.show()

### 3.8 TV Show Seasons Analysis

In [ ]:
tv_df = df[df['type'] == 'TV Show'].dropna(subset=['duration_seasons'])
seasons_counts = tv_df['duration_seasons'].value_counts().sort_index()

plt.figure(figsize=(12, 6))
sns.barplot(x=seasons_counts.index, y=seasons_counts.values, palette='dark:#E50914', hue=seasons_counts.index, legend=False)
plt.title('Distribution of TV Show Seasons', fontsize=16, color='white', weight='bold')
plt.xlabel('Number of Seasons', fontsize=12, color='white')
plt.ylabel('Count', fontsize=12, color='white')
plt.savefig('../dashboard_images/tv_seasons_dist.png', bbox_inches='tight', facecolor='#121212')
plt.show()

### 3.9 Oldest & Newest Content

In [ ]:
print("=== OLDEST 5 TITLES ON NETFLIX ===")
print(df[['title', 'type', 'release_year', 'country']].sort_values(by='release_year').head(5).to_string(index=False))

print("\n=== NEWEST 5 TITLES ON NETFLIX (BY RELEASE YEAR) ===")
print(df[['title', 'type', 'release_year', 'country']].sort_values(by='release_year', ascending=False).head(5).to_string(index=False))